In [ ]:
import pandas as pd
import datetime

try:
    df = pd.read_csv("scrobbles.csv")
except FileNotFoundError:
    print("Error: scrobbles.csv file not found.")
    exit()
    
df.head()

This dataset contains my personal music listening history collected from Last.fm, where each row represents one track scrobble.
Surprisingly, I have tracked my scrobbles since 2018

In [ ]:
df.shape
df.columns
df.info()

Getting df.info() tells me there are 92,700 entries, or in this case scrobbles, and that I have 7 columns, most of which are object datatypes.
In order to do what I want to do, I want to convert the utc_time column into datetime objects. (I can do this with pandas!)

In [ ]:
df['datetime'] = pd.to_datetime(df['uts'], unit='s')
df.dtypes
df.head(1)

Now that I have my datetime column for each entry, I want to restructure my table.
1) I want to drop unnecessary columns using df.drop(column) 
2) I can extract day, month and year from these datetime objects and make those new columns
3) I want to move the datetime and respective columns from the end to the beginning.

In [ ]:
df = df.drop(columns=['uts', 'utc_time', 'artist_mbid', 'album_mbid', 'track_mbid'])

df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day

df = df[
    [
        'datetime',
        'year',
        'month',
        'day',
        'track',
        'artist',
        'album' ]
]
df.head(1)

Beautiful! Now I have the exact datetimes of my scrobbles (including day, month, and year), artist, album, and song.

I now can filter and sort for whichever year or month I want.

In [ ]:
df_sorted = df.sort_values('datetime')

first_streams = df_sorted.groupby('year').first()

first_streams['datetime']

It looks like I took a break from scrobbling consistently until 2023. Let's filter my values for only 2023-2026. My last 3 years of scrobbles!

In [ ]:
df_analysis = df[(df['year'] >= 2023) & (df['year'] <= 2026)]

!!!! FUNCTIONS !!!!

In [ ]:
def filter_year(year):
    """
    Return a filtered dataframe containing only scrobbles from the given year.
    """
    data = df_analysis[df_analysis['year'] == year]

    if data.empty:
        print("No data found for that year.")
    
    return data

In [ ]:
def top_artists(year):
    """
    Display the top 3 most played ARTISTS for a given year.
    """
    data = filter_year(year)

    #count artists and return the top 3
    results = (data['artist'].value_counts().head(3))

    print("\nTop 3 Artists in", year)
    print(results)

In [ ]:
def top_albums(year):
    """
    Display the top 3 most played ALBUMS for a given year.
    """
    data = filter_year(year)

    #count values and return the top 3
    results = (data['album'].value_counts().head(3))

    print("\nTop 3 Albums in", year)
    print(results)    

In [ ]:
def top_songs(year):
    """
    Display the top 3 most played SONGS for a given year.
    """
    data = filter_year(year)

    #count TRACKS and return the top 3

    results = (data['track'].value_counts().head(3))

    print("\nTop 3 Songs in", year)
    print(results)    

#an idea for a future iteration, how can I ensure I'm not 
# counting different songs by the same name? 

In [ ]:
while True:

    print("\nWelcome to Corey's Spotify Wrapped. Which category would you like to see?")
    print("1 - Top 3 Artists")
    print("2 - Top 3 Albums")
    print("3 - Top 3 Songs")
    print("4 - Exit")

    choice = input("Choose an option: ")

    if choice == "4":
        print("Happy Spotify Wrapped!")
        break

    try:
        year = int(input("Enter a year (2023-2026): "))

        if year < 2023 or year > 2026:
            raise ValueError("Year out of range")

    except ValueError:
        print("Please enter a valid year between 2023 and 2026.")
    
        continue

    if choice == "1":
        top_artists(year)

    elif choice == "2":
        top_albums(year)

    elif choice == "3":
        top_songs(year)

    else:
        print("Invalid option")